# 4장 Observability 계약 확인

## Goal

공통 telemetry policy와 Risk API의 app-owned metric 계약을 구분하고, Risk API가 내보내는 metric 이름과 Grafana dashboard query가 일치하는지 확인합니다. Grafana Cloud 가입, Secret 설정, Alloy 실행과 dashboard import는 README에서 수행합니다.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "tta-aiqa repository root를 찾지 못했습니다."
ROOT

PosixPath('/Users/seungbaeji/Workspace/tta/tta-aiqa')

In [2]:
import json
import re
import pandas as pd
import yaml

platform_policy = yaml.safe_load((ROOT / "configs/observability/telemetry.yaml").read_text())
api_config = yaml.safe_load((ROOT / "configs/serving/api.yaml").read_text())
dashboard = json.loads((ROOT / "deploy/grafana-cloud/dashboards/ai-quality.json").read_text())
assert set(platform_policy) == {"schema_version", "service_namespace", "logging"}
declared_metrics = set(api_config["observability"]["metrics"].values())
queries = [
    target.get("expr", "")
    for panel in dashboard["panels"]
    for target in panel.get("targets", [])
]
query_text = "\n".join(queries)

## Steps

### 1. Dashboard panel과 datasource 확인

In [3]:
panels = pd.DataFrame([
    {
        "title": panel["title"],
        "type": panel["type"],
        "datasources": sorted({
            target.get("datasource", {}).get("type", "")
            for target in panel.get("targets", [])
        }),
    }
    for panel in dashboard["panels"]
])
panels

,title,type,datasources
0,Request rate,timeseries,[]
1,Error rate,timeseries,[]
2,P95 latency,timeseries,[]
3,Positive mortality-risk rate,timeseries,[]
4,Risk score P95,timeseries,[]
5,Missing features P95,timeseries,[]
6,Recent Risk API logs,logs,[]
7,Risk API traces,traces,[]


### 2. Metric query 일치 확인

In [4]:
metric_usage = pd.DataFrame([
    {"metric": metric, "used_by_dashboard": metric in query_text}
    for metric in sorted(declared_metrics)
])
metric_usage

,metric,used_by_dashboard
0,aiqa_risk_missing_features,True
1,aiqa_risk_predictions_total,True
2,aiqa_risk_request_duration_seconds,True
3,aiqa_risk_requests_total,True
4,aiqa_risk_score,True


## Checks

In [5]:
datasource_types = {
    panel.get("datasource", {}).get("type")
    for panel in dashboard["panels"]
}
assert dashboard["uid"] == "tta-aiqa-quality"
assert all(metric in query_text for metric in declared_metrics)
assert {"prometheus", "loki", "tempo"} <= datasource_types
assert "model_version" in query_text
print("Dashboard and telemetry contract checks passed.")

Dashboard and telemetry contract checks passed.


## Next Steps

개인 Grafana Cloud datasource UID를 설정하고 importer를 실행한 뒤 baseline traffic과 Candidate B traffic이 같은 dashboard에 서로 다른 `model_version`으로 누적되는지 확인합니다.